<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 1 — Is This Signal, or Noise? (Lancaster Quay Tides)</div>
</div>

#### The opening hook for Lecture 1a, and the dataset for the pandas/matplotlib warm-up in the Week 1 lab.

Source: Rowland Adams, J., Newman, J. & Stefanovska, A. (2023), *"Distinguishing between deterministic oscillations and noise,"* Eur. Phys. J. Spec. Top. 232, 3435–3457. The tidal case study in that paper (their Fig. 4) uses water-level readings from **Lancaster Quay**, on the River Lune — a gauge a five-minute walk from campus.

**How to use this notebook:** run Section 1 and *stop*. Look at the plot before reading Section 2. The point of the exercise is to ask the question honestly before being told the answer.

## <span style="color:#B5121B">1. A real record, no context yet</span>

Below is a genuine water-level record from a real gauge — the actual station this notebook is set up to pull from is Lancaster Quay (Environment Agency station notation `724735`, measure `724735-level-stage-i-15_min-m`, 15-minute readings, metres). No further description yet.

**Before running Section 2: is this signal, or is this noise?**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Data: read from a local datasets/ folder if you've cloned/downloaded the
# repository, otherwise straight from GitHub (e.g. in Colab). Same file either way.
DATA_URL = "https://raw.githubusercontent.com/MatSmithAstro/phys7101_2026_resources/main/workshops/week1/datasets/"

def data_path(filename):
    local = os.path.join("datasets", filename)
    return local if os.path.exists(local) else DATA_URL + filename



def load_cached_lancaster_quay(path=None):
    # The default. A real extract from the Environment Agency feed below,
    # fetched once (24 Sep - 2 Oct 2026) and saved so every student loads the
    # same numbers, and results are reproducible year to year.
    df = pd.read_csv(path or data_path("lancaster_quay_tidal.csv"))
    df["time"] = pd.to_datetime(df["time"])
    return df.sort_values("time").reset_index(drop=True)


def fetch_lancaster_quay(hours=192, timeout=5):
    # Real data loader -- Environment Agency flood-monitoring API, Lancaster
    # Quay gauge (station 724735, River Lune). Live/historical data, used
    # only if you want a fresher pull than the cached file above; it is NOT
    # reached from every environment (some sandboxes block outbound
    # requests), which is why there's a further fallback below.
    import requests
    from datetime import datetime, timedelta, timezone

    since = (datetime.now(timezone.utc) - timedelta(hours=hours)).strftime("%Y-%m-%dT%H:%M:%SZ")
    url = (
        "https://environment.data.gov.uk/flood-monitoring/id/measures/"
        "724735-level-stage-i-15_min-m/readings"
    )
    resp = requests.get(url, params={"since": since, "_limit": 10000}, timeout=timeout)
    resp.raise_for_status()
    items = resp.json()["items"]
    df = pd.DataFrame(items)[["dateTime", "value"]]
    df["dateTime"] = pd.to_datetime(df["dateTime"])
    df = df.rename(columns={"dateTime": "time", "value": "level_m"}).sort_values("time")
    df = df.reset_index(drop=True)
    return df


def synthetic_lancaster_quay(hours=192, seed=7):
    # SYNTHETIC PLACEHOLDER -- last-resort fallback, only used if neither the
    # cached file nor the live API above is reachable. Mimics the real
    # record's known properties rather than inventing arbitrary numbers:
    #   - M2 tidal period (~12h 25m), the dominant real-world tidal constituent
    #   - a non-sinusoidal shape (harmonics), as the paper's own model for
    #     this gauge assumes: X(t) = A(t) h(theta(t))
    #   - a sensor floor at 2.15 m (below this, the real gauge simply can't
    #     read a level) -- the paper notes this explicitly for this station
    #   - slow (multi-day) amplitude modulation, i.e. spring/neap-style
    #     variation
    rng = np.random.default_rng(seed)
    n = hours * 4  # 15-minute steps
    t_hours = np.arange(n) * 0.25
    period = 12.42  # M2 constituent, hours
    theta = 2 * np.pi * t_hours / period

    # slow spring/neap-style amplitude modulation (~14-day cycle)
    amplitude = 1.5 + 0.3 * np.sin(2 * np.pi * t_hours / (14 * 24) + 0.7)

    # non-sinusoidal shape: fundamental + a couple of harmonics, matching the
    # paper's description of the real Lancaster Quay record
    shape = np.sin(theta) + 0.25 * np.sin(2 * theta + 0.4) + 0.10 * np.sin(3 * theta + 1.1)

    baseline = 4.2
    noise = rng.normal(0, 0.06, size=n)
    level = baseline + amplitude * shape + noise

    # the real gauge can't read below 2.15 m — clip, don't pretend otherwise
    level = np.clip(level, 2.15, None)

    start = pd.Timestamp.now("UTC").floor("15min") - pd.Timedelta(hours=hours)
    time = start + pd.to_timedelta(t_hours, unit="h")
    return pd.DataFrame({"time": time, "level_m": level})


try:
    tide = load_cached_lancaster_quay()
    source = f"real (cached extract, captured up to {tide['time'].max():%Y-%m-%d %H:%M} UTC)"
except Exception as exc:
    try:
        tide = fetch_lancaster_quay(hours=192)
        source = "real (Environment Agency API, live)"
    except Exception as exc2:
        tide = synthetic_lancaster_quay(hours=192)
        source = f"SYNTHETIC placeholder (no cached file, and live API not reached: {exc2!r})"

print(f"Data source: {source}")
print(f"{len(tide)} readings, {tide['time'].min()} to {tide['time'].max()}")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(tide["time"], tide["level_m"], lw=1)
ax.set_xlabel("Time")
ax.set_ylabel("Water level (m)")
ax.set_title("Lancaster Quay, River Lune — raw record")
fig.tight_layout()
plt.show()

## <span style="color:#B5121B">2. The reveal</span>

This is tidal. The River Lune is tidal at Lancaster Quay, and the water level oscillates through two highs and two lows a day — the M2 lunar semi-diurnal constituent, period ≈ 12 h 25 min.

Look again at the raw trace in Section 1 without that knowledge, though, and it's easy to see why a careless analysis could write large stretches of it off as noise: the shape isn't a clean sine wave (there's real harmonic content — the tide here isn't symmetric), the amplitude drifts from day to day (spring/neap variation), and a naive frequency-domain summary can smear all of that into something that looks broadband and noise-like. That's exactly the failure mode Rowland Adams, Newman & Stefanovska (2023) set out to fix for this and two other real datasets in their paper: a classical power-spectral-density approach can mischaracterise a genuinely deterministic, physically-understood oscillation as noise, where a time–frequency approach (wavelet transform) correctly identifies it.

Their own preliminary model for this exact gauge is

$$X(t) = A(t)\, h(\theta(t)), \qquad \dot\theta(t) = \frac{2\pi}{12\,\text{h}},$$

a slowly-varying amplitude $A(t)$ riding a fixed-frequency phase $\theta(t)$, with $h$ a non-sinusoidal periodic function (hence the harmonics). You don't need to do anything with this equation yet — it's here so the shape of the synthetic fallback above isn't arbitrary, and because Week 4b will come back to exactly this kind of decomposition once the DFT is on the table.

## <span style="color:#B5121B">3. Describing the data</span>

The same descriptive-statistics toolkit from this week's lecture, applied to something real rather than a generic tutorial column.

In [ ]:
level = tide["level_m"]

print(f"n readings        : {len(level)}")
print(f"mean level         : {level.mean():.3f} m")
print(f"median level       : {level.median():.3f} m")
print(f"std deviation      : {level.std():.3f} m")
print(f"5th / 95th pctile  : {level.quantile(0.05):.3f} m / {level.quantile(0.95):.3f} m")
print(f"minimum reading    : {level.min():.3f} m")

near_min = (level <= level.min() + 0.02).sum()
print(f"\nreadings within 2 cm of that minimum: {near_min} "
      f"({100 * near_min / len(level):.1f}% of the record)")
print("(mean and median are a long way apart here -- a first hint that this")
print(" isn't a simple symmetric distribution. More on that below.)")

Those numbers summarise the data with single values — a mean, a spread. But a summary statistic can hide a lot about the *shape* of the data behind it. This is an **empirical distribution**: the actual pattern of values this real instrument produced, as opposed to a **theoretical distribution** (like the Gaussian/normal curve) that you might *assume* describes it without checking. Plot a histogram and look.

In [ ]:
from scipy.stats import skew, kurtosis

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(level, bins=40, color="#B5121B", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Water level (m)")
ax.set_ylabel("Count")
ax.set_title("Empirical distribution of tidal level readings")
fig.tight_layout()
plt.show()

print(f"skewness        : {skew(level):.2f}  "
      "(0 = symmetric like a Gaussian; positive = long tail toward higher values)")
print(f"excess kurtosis : {kurtosis(level):.2f}  "
      "(0 = Gaussian-like tails; negative = flatter-topped, less peaked)")

This is nowhere close to the bell-shaped curve you'd get by default if you assumed "normally distributed" without looking. Two things stand out:

- **A sharp pile-up on the low side.** Around half of all readings (378 of 766, in this cached extract) sit within about 2 cm of 2.22–2.23 m. That's not measurement noise — the gauge genuinely spends roughly half of every tidal cycle sitting at, or very near, one fixed low level, rather than continuing to sweep downward the way a clean sine wave would.
- **A long tail toward higher levels.** The remaining readings are spread out between that low level and the flood peaks (up to ~5.5 m), giving the distribution a clear **positive skew** (skewness ≈ +1.3) rather than the near-zero skew of a Gaussian.

Why would a real tidal signal look like this rather than symmetric? This gauge is on the River Lune at Lancaster — a *river* gauge, well upstream of the open sea, not a coastal one. A natural question to put to the room: a tidal estuary can have its low-water level held up by something other than the tide itself (the river's own base flow, a weir or sill downstream, channel geometry) while the rising tide is still perfectly free to push the surface up well above that floor. That would explain exactly this shape — a plateau at the bottom, a long tail at the top — but it's an inference from the data, not a fact read off a map, and worth treating as a discussion point rather than a stated conclusion (if you know this stretch of the Lune, you may know the real answer already).

One thing the shape doesn't call into question: the record is still genuinely tidal. The peaks in the raw trace above land on average every ~12.3 hours — close to the real M2 lunar semi-diurnal period (12 h 25 min) that drives most of the world's tides.

The lesson to take forward either way: a distribution's *shape* is information in its own right, and "probably roughly Gaussian" is an assumption to check with a histogram, not a default to reach for. You'll use empirical distributions like this one again — bootstrapping builds one directly in Week 5, and Week 7's Bayesian posteriors are distributions built from evidence in essentially the same spirit.

## <span style="color:#FD9029">4. Optional, ungraded: eyeball the period</span>

No FFT yet — that's Week 4b. Just look at a single day, zoomed in, and count.

In [ ]:
one_day = tide[tide["time"] < tide["time"].min() + pd.Timedelta(hours=48)]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(one_day["time"], one_day["level_m"], marker="o", ms=3, lw=1)
ax.set_xlabel("Time")
ax.set_ylabel("Water level (m)")
ax.set_title("Zoomed in: first 48 hours")
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

# A rough, by-eye-style peak count — nothing more sophisticated than that.
from scipy.signal import find_peaks
peak_idx, _ = find_peaks(one_day["level_m"].to_numpy(), distance=4 * 8)  # >=8h apart
peak_times = one_day["time"].to_numpy()[peak_idx]
if len(peak_times) >= 2:
    gaps_hours = np.diff(peak_times).astype("timedelta64[m]").astype(float) / 60
    print(f"{len(peak_times)} peaks found in 48 h; gaps between them (hours): "
          f"{np.round(gaps_hours, 2)}")
    print(f"rough estimated period: {gaps_hours.mean():.2f} h "
          f"(real M2 period is 12.42 h)")
else:
    print(f"{len(peak_times)} peak(s) found — widen the window to see a full cycle.")

## Closing the loop

Hold onto this parallel for a moment: the tide here is a real, small-relative-to-the-noise-floor signal that a careless analysis can miss entirely — and the catastrophic-cancellation demonstration later this session (`week1_foundations.ipynb`) is the same shape of problem one level down, a real, small *quantity* that a careless *calculation* destroys. Different scale, same lesson: numerical and analytical care isn't optional extra rigour, it's the difference between seeing what's really there and not.

This tidal dataset isn't used for the catastrophic-cancellation demo itself — that one deliberately needs a far more extreme scale mismatch than anything in this record, so it stays on its own synthetic large-offset dataset in `week1_foundations.ipynb`.

## Appendix: getting the real data

This notebook now loads real data by default, from a cached file rather than a live network call:

- **`datasets/lancaster_quay_tidal.csv` (what this notebook uses by default):** a genuine extract from the Environment Agency feed below, fetched once (24 Sep – 2 Oct 2026) and saved in this week's `datasets/` folder in the module repository. Every student gets identical numbers and results don't drift year to year. In Colab it's read directly from GitHub; in a local copy of the repository it's read from disk with no network needed — the same reasoning as `isotope_masses.csv` and `pandas_practice_dataset.csv` elsewhere in Week 1.
- **Live feed (`fetch_lancaster_quay`, used only if the cached file can't be read):** `https://environment.data.gov.uk/flood-monitoring/id/measures/724735-level-stage-i-15_min-m/readings` — station 724735, "Lancaster Quay", River Lune. Free, no API key, standard JSON. Useful if you want a fresher pull than the cached extract, but not needed to run this notebook.
- **Synthetic placeholder (`synthetic_lancaster_quay`, last resort):** only used if neither of the above is available — e.g. a fully offline sandbox with no cached file present. Labelled clearly in the printed output so it's never mistaken for real data.

**To refresh the cached file** (e.g. for a future year, or a longer record): run `fetch_lancaster_quay(hours=192).to_csv("lancaster_quay_tidal.csv", index=False)` once, anywhere with normal network access, then replace `workshops/week1/datasets/lancaster_quay_tidal.csv` in the module repository with the result and push.

**A separate, longer archived record exists but isn't what this notebook uses.** The paper this exercise is loosely inspired by cites a specific archived series (16–25 Feb 2022) at the Lancaster University research-data repository, `https://doi.org/10.17635/lancaster/researchdata/609`, for citation/provenance purposes. That record ships as part of a 16.3 GB `Data_and_code.zip` bundle, not a small standalone file — not practical to ask every student to download for a one-week workshop, which is why the cached extract above exists instead. Mentioned here only so the connection to the published paper isn't lost, not as something you need to fetch.

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis &middot; Lancaster University
</div>